# Challenge Sprint 3 — Estatística com Python

**Integrantes:** PREENCHER NOMES COMPLETOS E MATRÍCULAS  
**Ambiente:** Google Colab  
**Base:** renda e gasto de 50 famílias, conforme exemplo didático da Alura.

## 1. Preparação do ambiente e carregamento da base
Execute a célula abaixo e selecione o arquivo `dados_familias.csv` quando solicitado.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import norm
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

arquivo = Path('/content/dados_familias.csv')
if not arquivo.exists():
    from google.colab import files
    enviados = files.upload()
    arquivo = Path(next(iter(enviados)))

dados = pd.read_csv(arquivo)
dados.head()

In [ ]:
# Validação da base
colunas_obrigatorias = {'renda_familiar', 'gasto_familiar'}
assert colunas_obrigatorias.issubset(dados.columns), 'A base não possui as colunas esperadas.'
assert not dados[list(colunas_obrigatorias)].isna().any().any(), 'Há valores ausentes.'
print(f'{len(dados)} registros carregados com sucesso.')
dados[['renda_familiar', 'gasto_familiar']].describe()

## 2. Probabilidade acima da mediana
Assumimos que o gasto familiar segue uma Distribuição Normal com média e desvio padrão estimados pela amostra. Calculamos `P(X > mediana)` com a função de sobrevivência `norm.sf`.

In [ ]:
def classificar_evento(probabilidade):
    if probabilidade <= 0.05:
        return 'raro'
    if probabilidade <= 0.25:
        return 'pouco provável'
    if probabilidade <= 0.75:
        return 'provável'
    return 'quase certo'

gastos = dados['gasto_familiar']
media = gastos.mean()
mediana = gastos.median()
desvio = gastos.std(ddof=1)
prob_acima_mediana = norm.sf(mediana, loc=media, scale=desvio)

print(f'Mediana: R$ {mediana:,.2f}')
print(f'P(X > mediana): {prob_acima_mediana:.2%}')
print(f'Classificação: {classificar_evento(prob_acima_mediana)}')

**Interpretação:** a probabilidade calculada é aproximadamente 44,36%, portanto o evento é classificado como **provável**. O resultado não é exatamente 50% porque foi usada a mediana amostral como ponto de corte em uma Normal parametrizada pela média e pelo desvio padrão amostrais.

## 3. Probabilidade no intervalo média ± 2s
O intervalo é calculado por `[média − 2s, média + 2s]`. A probabilidade resulta da diferença entre as probabilidades acumuladas nos dois limites.

In [ ]:
limite_inferior = media - 2 * desvio
limite_superior = media + 2 * desvio
prob_intervalo = (
    norm.cdf(limite_superior, loc=media, scale=desvio)
    - norm.cdf(limite_inferior, loc=media, scale=desvio)
)

print(f'Média: R$ {media:,.2f}')
print(f'Desvio padrão amostral: R$ {desvio:,.2f}')
print(f'Intervalo: [R$ {limite_inferior:,.2f}; R$ {limite_superior:,.2f}]')
print(f'Probabilidade: {prob_intervalo:.2%}')
print(f'Classificação: {classificar_evento(prob_intervalo)}')

In [ ]:
eixo_x = np.linspace(media - 4 * desvio, media + 4 * desvio, 800)
densidade = norm.pdf(eixo_x, loc=media, scale=desvio)
fig, ax = plt.subplots(figsize=(11, 5.5))
ax.plot(eixo_x, densidade, color='#174A7E', linewidth=2.4, label='Distribuição Normal ajustada')
ax.fill_between(eixo_x, densidade, where=(eixo_x >= limite_inferior) & (eixo_x <= limite_superior), color='#5CB8B2', alpha=0.38, label='Intervalo média ± 2s')
ax.fill_between(eixo_x, densidade, where=eixo_x >= mediana, color='#F2A65A', alpha=0.28, label='Área acima da mediana')
ax.axvline(media, color='#174A7E', linestyle='--', label=f'Média = R$ {media:,.2f}')
ax.axvline(mediana, color='#B45309', linestyle=':', label=f'Mediana = R$ {mediana:,.2f}')
ax.set(title='Distribuição Normal do gasto familiar', xlabel='Gasto familiar (R$)', ylabel='Densidade de probabilidade')
ax.legend(frameon=False)
ax.grid(axis='y', alpha=0.2)
plt.show()

**Interpretação:** cerca de 95,45% dos valores da Normal estão entre R$ 376,69 e R$ 3.645,55. Por superar 75%, esse evento é classificado como **quase certo**.

## 4. Regressão Linear Simples
O modelo estima o gasto familiar (variável dependente) a partir da renda familiar (variável explicativa).

In [ ]:
X = dados[['renda_familiar']]
y = dados['gasto_familiar']
modelo = LinearRegression().fit(X, y)
previsoes = modelo.predict(X)
intercepto = modelo.intercept_
coeficiente = modelo.coef_[0]
r2 = r2_score(y, previsoes)
rmse = np.sqrt(mean_squared_error(y, previsoes))

print(f'Equação: gasto = {intercepto:.4f} + {coeficiente:.4f} × renda')
print(f'R²: {r2:.4f}')
print(f'RMSE: R$ {rmse:,.2f}')

In [ ]:
ordem = np.argsort(dados['renda_familiar'].to_numpy())
x_ordenado = dados['renda_familiar'].to_numpy()[ordem]
x_previsao = pd.DataFrame({'renda_familiar': x_ordenado})
fig, ax = plt.subplots(figsize=(11, 6))
ax.scatter(X['renda_familiar'], y, color='#5CB8B2', edgecolor='white', s=58, label='Famílias observadas')
ax.plot(x_ordenado, modelo.predict(x_previsao), color='#C33C54', linewidth=2.5, label='Reta ajustada')
ax.set(title='Regressão linear: gasto familiar em função da renda', xlabel='Renda familiar (R$)', ylabel='Gasto familiar (R$)')
ax.legend(frameon=False)
ax.grid(alpha=0.2)
plt.show()

### Interpretação dos coeficientes

- **Intercepto (207,9033):** gasto estimado pelo modelo quando a renda é zero. É o ponto matemático em que a reta cruza o eixo vertical e deve ser interpretado com cautela, pois a amostra não contém renda igual a zero.
- **Coeficiente angular (0,2973):** para cada aumento de R$ 1,00 na renda, o gasto familiar esperado aumenta cerca de R$ 0,2973. De modo equivalente, R$ 1.000 adicionais de renda estão associados a aproximadamente R$ 297,31 adicionais de gasto.
- **R² (0,9699):** cerca de 96,99% da variação observada nos gastos é explicada linearmente pela renda nesta amostra. A associação não prova causalidade.
- **RMSE (R$ 140,36):** o erro típico das previsões dentro da amostra é de aproximadamente R$ 140,36.

## 5. Conclusão
A estatística descritiva e a Distribuição Normal quantificaram a chance dos eventos definidos no enunciado. A regressão linear transformou a relação entre renda e gasto em um modelo preditivo interpretável. Juntas, essas técnicas mostram como a estatística sustenta o aprendizado de máquina: primeiro descrevemos a incerteza dos dados e depois estimamos uma relação capaz de produzir previsões.

## Referências
- Alura. *Cálculo da probabilidade da distribuição normal com quaisquer valores de média e desvio padrão*.
- Alura. *Estatística com Python: Correlação e Regressão*.